# VeritasAI · Approach D · End-to-end fine-tune + deployment
EfficientNet-B0 with its last 3 stages trainable + temporal attention head, AMP fp16. Then: efficiency benchmark, ONNX export, results table + plot, checkpoint for Django. GPU: T4, ~30-60 min.

In [ ]:
# ==== 0. SETUP (same in every notebook) ===================================
# Kaggle: Settings -> Accelerator = GPU T4 x2 (or None for notebook 00), Internet = ON
# Colab : Runtime -> Change runtime type -> T4 GPU
import os, sys
KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if KAGGLE else "/content"
REPO_URL = "https://github.com/Bhoomified/VeritasAI.git"   # your repo (push the new ml/ files first!)
os.chdir(WORK)
if not os.path.exists("VeritasAI"):
    os.system(f"git clone -q {REPO_URL}")
os.chdir(f"{WORK}/VeritasAI"); os.system("git pull -q")
os.system('pip -q install "opencv-python-headless==4.10.0.84" onnx onnxruntime tabulate')
if not KAGGLE:   # Colab: keep outputs on Google Drive so a disconnect never loses work
    from google.colab import drive; drive.mount("/content/drive")
STORE = "/kaggle/working" if KAGGLE else "/content/drive/MyDrive/veritas"
os.makedirs(STORE, exist_ok=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
# ==== where the face strips from notebook 00 live ==========================
# Kaggle: "Add Input" -> your notebook-00 output; it appears under /kaggle/input/<name>/
FACES = "/kaggle/input/veritas-faces/faces" if KAGGLE else f"{STORE}/faces"
TRAIN_SET, CROSS_SET = "ffpp", "celebdf"          # train on FF++, test unseen on Celeb-DF
F = f"{FACES}/{TRAIN_SET}"
TR, VA, TE = f"{F}/{TRAIN_SET}_train.csv", f"{F}/{TRAIN_SET}_val.csv", f"{F}/{TRAIN_SET}_test.csv"
CROSS = f"{FACES}/{CROSS_SET}/{CROSS_SET}_test.csv"
RESULTS = f"{STORE}/results.csv"; CK = f"{STORE}/checkpoints"

In [ ]:
!python -m ml.finetune --mode video --backbone efficientnet_b0 --head attn --unfreeze 3 --clip 8 --epochs 8 --bs 8 \
   --faces {F} --train {TR} --val {VA} --test {TE} --cross {CROSS} --out {CK} --results {RESULTS} --workers 4

In [ ]:
# ==== efficiency table (run on CPU numbers = deployment reality) =============
!python -m ml.benchmark --backbones mobilenet_v3_small mobilenet_v3_large efficientnet_b0 resnext50_32x4d
!python -m ml.benchmark --checkpoint {CK}/efficientnet_b0_fusion.pt --onnx {STORE}/veritas.onnx

In [ ]:
# ==== paper table + accuracy-vs-cost plot ==================================
!python -m ml.report --results {RESULTS} --out {STORE}/report
from IPython.display import Image; Image(f"{STORE}/report/auc_vs_latency.png")

Download the best `.pt` from `checkpoints/` → put it in `VeritasAI/checkpoints/` on your laptop → set `MODEL_CHECKPOINT_PATH` in `.env` → `python manage.py runserver`.